# Fine-tuning AraT5 — Dialect → MSA
Notebook مستقل بالكامل: بيقرا الداتا محليًا (output ملف `DataPreparation.ipynb`)، يدرب موديل **UBC-NLP/AraT5-base** على الـ GPU (CUDA)، يحفظه محليًا، وبرضو بيحفظ predictions بتاعته على الـ test set في فولدر مشترك عشان ملف المقارنة الأخير يستخدمها.


In [ ]:
# 3) Config
import os

# Local paths (Local machine) - Change BASE_DIR if you want a different location
DATA_DIR        = r'C:\Users\youssef\Desktop\NLP Project\Arabic MT\files\MADAR\data'
BASE_DIR        = os.path.dirname(DATA_DIR)   # Parent directory of the data folder
MODELS_DIR      = os.path.join(BASE_DIR, 'models')
PREDICTIONS_DIR = os.path.join(BASE_DIR, 'predictions')   # Each model saves its test set predictions here

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(PREDICTIONS_DIR, exist_ok=True)

DATASET_FILE = 'regMADARmsa.csv'
DATA_PATH = os.path.join(DATA_DIR, DATASET_FILE)

MODEL_NAME = 'UBC-NLP/AraT5-base'
OUTPUT_SUBFOLDER = 'arat5_reg'
MODEL_OUTPUT_DIR = os.path.join(MODELS_DIR, OUTPUT_SUBFOLDER)
os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)

MAX_SOURCE_LEN = 64
MAX_TARGET_LEN = 64
NUM_EPOCHS = 30
BATCH_SIZE = 16
LEARNING_RATE = 0.0001
SEED = 42

print('Data path:', DATA_PATH)
print('Model output dir:', MODEL_OUTPUT_DIR)

Data path: C:\Users\youssef\Desktop\NLP Project\Arabic MT\files\MADAR\data\regMADARmsa.csv
Model output dir: C:\Users\youssef\Desktop\NLP Project\Arabic MT\files\MADAR\models\arat5_reg


In [2]:
# 4) Load data (Output of DataPreparation.ipynb -> Input here)
import pandas as pd
from datasets import Dataset, DatasetDict

df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=['0', '1']).reset_index(drop=True)
df = df.rename(columns={'0': 'source', '1': 'target'})
print('Total pairs:', len(df))
df.head()


c:\Users\youssef\anaconda3\envs\cuda_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total pairs: 50923


,source,target
0,لو سمحت إملأ إستمارة الحاجات الضايعة.,من فضلك أكمل البيانات الناقصة في التقرير .
1,يصير نتزلج في ماوناكيا في هالوقت؟,ألا يزال ممكناً ممارسة التزلج على الجليد في مو...
2,حولت الرحلة لخطوط دلتا رحلة اتنين واحد خمسه .,أحوّل إلى رحلة خطوط طيران دلتا رقم اثنان واحد ...
3,أيضا هم غالين.,هم أيضاً غاليين .
4,الأكترية شغل و شوي سياحة.,كثيرها عمل وقليلها تنزه .


In [ ]:
# 4b) Quick data quality / duplication check
print('Total rows:', len(df))
print('Unique source sentences:', df['source'].nunique())
print('Unique target sentences:', df['target'].nunique())
print('Unique (source, target) pairs:', df.drop_duplicates(subset=['source', 'target']).shape[0])
print('Duplicate source ratio: {:.1%}'.format(1 - df['source'].nunique() / len(df)))

df['source_len'] = df['source'].astype(str).str.split().str.len()
df['target_len'] = df['target'].astype(str).str.split().str.len()
print('\nSource length (words) stats:')
print(df['source_len'].describe())
print('\nTarget length (words) stats:')
print(df['target_len'].describe())
df = df.drop(columns=['source_len', 'target_len'])


Total rows: 50923
Unique source sentences: 50586
Unique target sentences: 10546
Unique (source, target) pairs: 50923
Duplicate source ratio: 0.7%

Source length (words) stats:
count    50923.000000
mean         6.227913
std          3.584271
min          1.000000
25%          4.000000
50%          5.000000
75%          8.000000
max         59.000000
Name: source_len, dtype: float64

Target length (words) stats:
count    50923.000000
mean         8.041239
std          4.126242
min          2.000000
25%          5.000000
50%          7.000000
75%         10.000000
max         59.000000
Name: target_len, dtype: float64


In [ ]:
# 5) Train / validation / test split (Use the same SEED across all notebooks so the test set remains identical for fair comparison)
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.10, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED)
print('Train:', len(train_df), '| Val:', len(val_df), '| Test:', len(test_df))

raw_datasets = DatasetDict({
    'train': Dataset.from_pandas(train_df.reset_index(drop=True)),
    'validation': Dataset.from_pandas(val_df.reset_index(drop=True)),
    'test': Dataset.from_pandas(test_df.reset_index(drop=True)),
})
raw_datasets


Train: 45830 | Val: 2546 | Test: 2547


DatasetDict({
    train: Dataset({
        features: ['source', 'target'],
        num_rows: 45830
    })
    validation: Dataset({
        features: ['source', 'target'],
        num_rows: 2546
    })
    test: Dataset({
        features: ['source', 'target'],
        num_rows: 2547
    })
})

In [ ]:
# 6) Tokenizer + Model
import numpy as np
import evaluate
import inspect
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

assert torch.cuda.is_available(), (
    'CUDA is not available! Make sure PyTorch is installed with CUDA support '
    'and that your GPU is detected (torch.cuda.is_available() == False).'
)

device = torch.device('cuda')
print('Using device:', device, '-', torch.cuda.get_device_name(0))

bleu_metric = evaluate.load('sacrebleu')

PREFIX = ''
TGT_LANG = None

tokenizer_kwargs_extra = {}
# This model does not use src_lang
# This model does not use tgt_lang

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, **tokenizer_kwargs_extra)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

# If the model supports forced_bos_token (e.g., NLLB/mBART), set it to the target language
if TGT_LANG:
    try:
        forced_bos_id = tokenizer.convert_tokens_to_ids(TGT_LANG)
        if forced_bos_id is not None and forced_bos_id != tokenizer.unk_token_id:
            model.config.forced_bos_token_id = forced_bos_id
    except Exception as e:
        print('Warning: Could not set forced_bos_token_id -', e)

Using device: cuda - NVIDIA RTX 2000 Ada Generation


Loading weights: 100%|██████████| 284/284 [00:00<00:00, 46123.38it/s]


In [ ]:
# 7) Preprocessing + Metrics
def preprocess(examples):
    inputs = [PREFIX + s for s in examples['source']]
    model_inputs = tokenizer(inputs, max_length=MAX_SOURCE_LEN, truncation=True)
    labels = tokenizer(text_target=examples['target'], max_length=MAX_TARGET_LEN, truncation=True)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tokenized_datasets = raw_datasets.map(
    preprocess,
    batched=True,
    remove_columns=raw_datasets['train'].column_names
)

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.asarray(preds)
    labels = np.asarray(labels)

    # If logits (3D) are returned instead of token IDs, take the argmax
    if preds.ndim == 3:
        preds = np.argmax(preds, axis=-1)

    vocab_size = len(tokenizer)

    # Replace any invalid token IDs to prevent OverflowError during decoding
    preds = np.where(
        (preds >= 0) & (preds < vocab_size),
        preds,
        tokenizer.pad_token_id
    ).astype(np.int64)

    labels = np.where(
        labels != -100,
        labels,
        tokenizer.pad_token_id
    ).astype(np.int64)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels_bleu = [[l.strip()] for l in decoded_labels]

    result = bleu_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels_bleu
    )

    return {'bleu': result['score']}

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

Map: 100%|██████████| 2547/2547 [00:00<00:00, 63186.96 examples/s]


In [ ]:
# 8) Training Arguments + Trainer
import torch

# fp16 may cause NaN issues with T5-family models (AraT5/mT5), so it is disabled.
# bf16 is more stable and is enabled only if the GPU supports it (e.g., A100, L4).
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

training_kwargs = dict(
    output_dir=MODEL_OUTPUT_DIR,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    weight_decay=0.01,
    num_train_epochs=NUM_EPOCHS,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    fp16=False,
    bf16=USE_BF16,
    label_smoothing_factor=0.1,
    load_best_model_at_end=True,
    metric_for_best_model='bleu',
    greater_is_better=True,
    report_to='none',
    logging_steps=50,
)
training_kwargs.update({})

args = Seq2SeqTrainingArguments(**training_kwargs)

# The parameter name changed across transformers versions:
# 'tokenizer' (older) -> 'processing_class' (newer)
trainer_sig = inspect.signature(Seq2SeqTrainer.__init__).parameters
tokenizer_kwarg = (
    {'processing_class': tokenizer}
    if 'processing_class' in trainer_sig
    else {'tokenizer': tokenizer}
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    **tokenizer_kwarg,
)

In [8]:
# 9) Train
trainer.train()


Epoch,Training Loss,Validation Loss,Bleu
1,4.107971,3.679975,23.310436
2,3.492774,3.156409,35.081834
3,3.163574,2.909077,39.604300
4,2.942118,2.745672,42.533798
5,2.782582,2.622193,45.947099
6,2.600807,2.533420,48.474522
7,2.510920,2.461636,50.951907
8,2.427659,2.393038,53.132420
9,2.336619,2.337967,55.325904
10,2.251834,2.297558,57.223144


That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.44it/s]
That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.05s/it]
That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.18it/s]
That's 100 lines that end in a tokenized period (

TrainOutput(global_step=85950, training_loss=2.341767302774426, metrics={'train_runtime': 24840.3929, 'train_samples_per_second': 55.349, 'train_steps_per_second': 3.46, 'total_flos': 3.565516481243136e+16, 'train_loss': 2.341767302774426, 'epoch': 30.0})

In [10]:
# 10) Evaluate on test set + save model
test_metrics = trainer.evaluate(tokenized_datasets['test'], metric_key_prefix='test')
print('Test metrics:', test_metrics)

trainer.save_model(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)

with open(os.path.join(MODEL_OUTPUT_DIR, 'test_metrics.txt'), 'w') as f:
    f.write(str(test_metrics))


That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


Training Loss,Validation Loss,Epoch,Bleu
1.818102,2.009576,30,71.564747


Test metrics: {'test_loss': 2.009575605392456, 'test_bleu': 71.56474701533628}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.64it/s]


In [ ]:
# 11) Generate + Save Predictions on the Test Set
# (These predictions will be used later in the comparison/voting notebook.)

import torch

model.to(device)  # Device was defined earlier (CUDA)
model.eval()

test_sources = test_df['source'].tolist()
test_targets = test_df['target'].tolist()

predictions = []
BATCH = 16

with torch.no_grad():
    for i in range(0, len(test_sources), BATCH):
        batch_src = [PREFIX + s for s in test_sources[i:i+BATCH]]
        enc = tokenizer(
            batch_src,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=MAX_SOURCE_LEN
        ).to(device)

        gen_kwargs = dict(max_length=MAX_TARGET_LEN)

        if TGT_LANG:
            try:
                gen_kwargs['forced_bos_token_id'] = tokenizer.convert_tokens_to_ids(TGT_LANG)
            except Exception:
                pass

        out_ids = model.generate(**enc, **gen_kwargs)
        decoded = tokenizer.batch_decode(out_ids, skip_special_tokens=True)
        predictions.extend(decoded)

pred_df = pd.DataFrame({
    'idx': test_df.index,
    'source': test_sources,
    'reference': test_targets,
    'prediction': predictions,
})

pred_path = os.path.join(PREDICTIONS_DIR, f'{OUTPUT_SUBFOLDER}_predictions.csv')
pred_df.to_csv(pred_path, index=False, encoding='utf-8-sig')

print('Saved predictions to:', pred_path)
pred_df.head()

Saved predictions to: C:\Users\youssef\Desktop\NLP Project\Arabic MT\files\MADAR\predictions\arat5_reg_predictions.csv


,idx,source,reference,prediction
0,35728,هذا مصنوع من القطن؟,هل هذا مصنوع من القطن ؟,هل هذا مصنوع من القطن ؟
1,50512,نفسى اتسدت.,لم يعد لي شهية .,أود أن أغلق على الباب .
2,34218,خلصنا من قبل .,لقد نفد منا للتو .,لقد انتهيت من ذلك من قبل .
3,28027,حلوة الاستضافة بالشامبانيا دي بعد الركوب، بصحتك.,أحب هذه الخدمة بتقديم شامبانيا للتحية بعد الإق...,جميل أن أراك في هذه الدورة بعد الإقلاع ، من فض...
4,17777,يوم الجمعة في الليل، الساعة تسع.,"مساء الجمعة , في التاسعة .","مساء الجمعة , في التاسعة ."


In [ ]:
def load_and_translate(model_subfolder, sentence, prefix=''):
    model_dir = os.path.join(MODELS_DIR, model_subfolder)
    if not os.path.isdir(model_dir):
        raise FileNotFoundError(f'Folder not found: {model_dir}\nMake sure training has completed and the model was actually saved.')

    tok = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(model_dir, local_files_only=True).to(
        'cuda' if __import__('torch').cuda.is_available() else 'cpu'
    )

    inputs = tok(prefix + sentence, return_tensors='pt', truncation=True, max_length=MAX_SOURCE_LEN).to(mdl.device)
    output_ids = mdl.generate(**inputs, max_length=MAX_TARGET_LEN)
    return tok.decode(output_ids[0], skip_special_tokens=True)

print(load_and_translate('arat5_reg', 'عامل ايه', prefix=PREFIX))

Loading weights: 100%|██████████| 281/281 [00:00<00:00, 7581.56it/s]


كيف حالك ؟
